# ROGII — `flat`
Thin offline notebook: install the wheel, run full CV, and write a local submission.


In [ ]:
from pathlib import Path
import os, subprocess, sys

MODEL_ID = "flat"
RUN_ID = "kaggle-flat"
CPU_WORKERS = 4
RUNTIME_ROOT = Path("/kaggle/input/datasets/danielwuu/upload/rogii_kaggle_dataset")
DATA_ROOT = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
WORK = Path("/kaggle/working")
RUN_ROOT = WORK / "runs" / RUN_ID

for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[variable] = str(CPU_WORKERS)
required = [
    RUNTIME_ROOT / "configs" / "folds" / "main_5fold.csv",
    DATA_ROOT / "train",
    DATA_ROOT / "test",
    DATA_ROOT / "sample_submission.csv",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Required Kaggle inputs are missing: {missing}")
wheel = next((RUNTIME_ROOT / "wheels").glob("rogii-*.whl"))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "--force-reinstall", str(wheel)],
    check=True,
)

run_command = [
    sys.executable, "-m", "rogii", "run",
    "--model", MODEL_ID,
    "--profile", "full",
    "--run-id", RUN_ID,
    "--data-root", str(DATA_ROOT),
    "--fold-config", str(RUNTIME_ROOT / "configs" / "folds" / "main_5fold.csv"),
    "--output-root", str(WORK / "runs"),
    "--workers", str(CPU_WORKERS),
]
subprocess.run(run_command, check=True)
subprocess.run([
    sys.executable, "-m", "rogii", "submit",
    "--run", str(RUN_ROOT), "--data-root", str(DATA_ROOT),
    "--output", str(WORK / "submission.csv"), "--workers", str(CPU_WORKERS),
], check=True)
print(f"submission={WORK / 'submission.csv'}")
